<img src="https://developer.download.nvidia.com/notebooks/dlsw-notebooks/rivaasrasr-basics/nvidia_logo.png" style="width: 90px; float: right;">

# How do I force an end-of-utterance from the client side?

This tutorial shows how to use the `force_eou` runtime flag to make the Riva ASR NIM finalize the current partial transcript on demand &mdash; without closing the stream &mdash; using the **Nemotron ASR Streaming** (cache-aware RNNT) NIM. <br>

## Why force end-of-utterance?

By default, Riva detects end-of-utterance (EOU) on the server using a silence-based timer (typically several hundred milliseconds of trailing silence before the final transcript is emitted). That works well, but a real-time agent often has *better information* than the server &mdash; for example:

- A push-to-talk button release
- A turn-taking signal from a conversational agent
- A client-side voice activity detector that is faster or more aggressive than the server's
- An LLM deciding "I have what I need, finalize now"

The `force_eou` flag is an extension point for these cases. The client sets it on a particular audio chunk; the server flushes the current utterance immediately, emits a final transcript, and **keeps the stream open** so subsequent audio continues as a new utterance.

In this tutorial we use a simple client-side silence detector (RMS energy) as a ***proxy*** for those higher-level signals so that the demo is fully self-contained.

> **Model support:** `force_eou` is currently honored only by **Nemotron ASR Streaming**. <br>This feature will not work with **Parakeet-CTC** and **Parakeet-RNNT** NIMs.

## Requirements and setup

1. **Start the Nemotron ASR Streaming NIM server.**
   Follow the [Riva NIM getting started guide](https://docs.nvidia.com/nim/speech/latest/about/index.html#launching-the-nim) to launch the NIM. Use:
   - `CONTAINER_ID = nemotron-asr-streaming`
   - `NIM_TAGS_SELECTOR = "name=nemotron-asr-streaming,mode=str"`

   The NIM exposes gRPC on port `50051` 

2. **Install the Riva Client library.**
   ```bash
   sudo apt-get install python3-pip
   pip install -U nvidia-riva-client
   ```

   This tutorial uses the `runtime_config` field on `StreamingRecognizeRequest` (proto `map<string, string>`).

#### Import the Riva client libraries

In [ ]:
import io
import time
import wave
from pathlib import Path

import numpy as np
import IPython.display as ipd

import riva.client
import riva.client.proto.riva_asr_pb2 as rasr
import riva.client.proto.riva_asr_pb2_grpc as rasr_grpc

#### Connect to the Riva NIM server

The default URI assumes a local NIM deployment on port `50051`. Adjust if your server is remote or behind a Helm chart.

In [ ]:
auth = riva.client.Auth(uri='localhost:50051')
asr_stub = rasr_grpc.RivaSpeechRecognitionStub(auth.channel)

## Choose a test audio sample

We use the existing `audio_samples/en-US_wordboosting_sample2.wav` clip, which contains two natural utterances separated by a brief mid-clip pause. The pause is short enough that the server's default endpointing threshold won't fire a mid-stream final on its own, making the effect of `force_eou` clearly visible.

In [ ]:
AUDIO_PATH = Path("audio_samples/en-US_wordboosting_sample2.wav")

with wave.open(str(AUDIO_PATH), 'rb') as w:
    params = w.getparams()

print(f"Sample rate    : {params.framerate} Hz")
print(f"Channels       : {params.nchannels}")
print(f"Total duration : {params.nframes / params.framerate:.2f} s")
ipd.Audio(str(AUDIO_PATH))

## Streaming recognition config

We use the same config for both runs (with and without `force_eou`) so the only variable is the client trigger. Server-side endpointing is left at its default; on Nemotron ASR Streaming the silence threshold is around 800 ms: long enough that a 200 ms client trigger will visibly preempt it.

In [ ]:
streaming_config = rasr.StreamingRecognitionConfig(
    config=rasr.RecognitionConfig(
        encoding=riva.client.AudioEncoding.LINEAR_PCM,
        sample_rate_hertz=params.framerate,
        audio_channel_count=1,
        language_code="en-US",
        max_alternatives=1,
        enable_automatic_punctuation=True,
    ),
    interim_results=True,
)

## Helper: collect responses with arrival timestamps

To compare runs, we record the wall-clock time at which each **final** transcript arrives, measured from the start of the streaming call. This lets us see how much earlier `force_eou` finalizes the first utterance.

In [ ]:
CHUNK_FRAMES = 1600  # 100 ms at 16 kHz; the silence detector buffers internally,
                     # so any chunk size is fine -- this just controls how often
                     # we hand a chunk to the gRPC stream.


def collect_finals(response_iterator, t0, force_eou_sends=None, max_match_delay_s=1.0):
    """Print partials inline and yield (elapsed_s, transcript) for each final.

    If `force_eou_sends` (a deque of wall-clock timestamps) is provided, each
    final is matched against the oldest pending send, and the delay is printed.
    Sends older than `max_match_delay_s` are aged out -- this handles spurious
    force_eou signals that the server silently dropped (e.g., when no speech
    was buffered yet).
    """
    for response in response_iterator:
        for result in response.results:
            if not result.alternatives:
                continue
            transcript = result.alternatives[0].transcript
            now = time.monotonic()
            elapsed = now - t0
            if result.is_final:
                delay_str = ""
                if force_eou_sends is not None:
                    while force_eou_sends and now - force_eou_sends[0] > max_match_delay_s:
                        force_eou_sends.popleft()
                    if force_eou_sends:
                        send_t = force_eou_sends.popleft()
                        delay_str = f" (+{(now - send_t) * 1000:.0f} ms after force_eou)"
                print(f"[final   t={elapsed:5.2f}s]{delay_str} {transcript!r}")
                yield elapsed, transcript
            else:
                print(f"[partial t={elapsed:5.2f}s] {transcript!r}")

## Run 1 &mdash; baseline (no `force_eou`)

The audio is streamed in real time (`sleep_audio_length` throttles the iterator). The server's own silence-based endpointing is the only thing that can produce mid-stream finals.

In [ ]:
def baseline_request_generator(streaming_config, chunks):
    yield rasr.StreamingRecognizeRequest(streaming_config=streaming_config)
    for chunk in chunks:
        yield rasr.StreamingRecognizeRequest(audio_content=chunk)


print("--- baseline run (no force_eou) ---")
with riva.client.AudioChunkFileIterator(
    str(AUDIO_PATH), CHUNK_FRAMES, delay_callback=riva.client.sleep_audio_length
) as chunks:
    t0 = time.monotonic()
    responses = asr_stub.StreamingRecognize(
        baseline_request_generator(streaming_config, chunks),
        metadata=auth.get_auth_metadata(),
    )
    baseline_finals = list(collect_finals(responses, t0))

## Helper: client-side silence detector

A minimal RMS-based detector. Audio chunks are split into 20 ms windows; we count consecutive low-RMS windows. After 200 ms of continuous silence following speech, we fire `force_eou=True` **once** for that silence event &mdash; subsequent silent chunks are suppressed until the next speech&rarr;silence transition. This matches how real applications would emit a single edge-triggered finalize signal.

In [ ]:
class SilenceDetector:
    """Edge-triggered silence detector. Fires once per speech->silence transition.

    Buffers audio internally so it works regardless of caller chunk size; the
    only requirement is that the analysis window (default 20 ms) fits within
    the audio stream as a whole.
    """

    def __init__(
        self,
        sample_rate: int,
        window_ms: int = 20,
        silence_threshold_ms: int = 200,
        rms_threshold: float = 300.0,
    ):
        self.window_ms = window_ms
        self.window_samples = int(sample_rate * window_ms / 1000)
        self.windows_needed = silence_threshold_ms // window_ms
        self.rms_threshold = rms_threshold
        self.consec_silent = 0
        self.windows_processed = 0  # tracks audio-time position
        self.in_speech = False
        self.fired_for_current_silence = False
        self.buffer = np.empty(0, dtype=np.int16)  # leftover samples between chunks

    def process_chunk(self, chunk_bytes: bytes) -> bool:
        new_samples = np.frombuffer(chunk_bytes, dtype=np.int16)
        samples = np.concatenate([self.buffer, new_samples])
        n_windows = len(samples) // self.window_samples
        fired = False
        for i in range(n_windows):
            self.windows_processed += 1
            window = samples[i * self.window_samples:(i + 1) * self.window_samples].astype(np.float32)
            rms = float(np.sqrt(np.mean(window ** 2)))
            if rms < self.rms_threshold:
                self.consec_silent += 1
            else:
                self.consec_silent = 0
                self.in_speech = True
                self.fired_for_current_silence = False
            if (
                self.in_speech
                and not self.fired_for_current_silence
                and self.consec_silent >= self.windows_needed
            ):
                self.fired_for_current_silence = True
                silence_ms = self.consec_silent * self.window_ms
                silence_end = self.windows_processed * self.window_ms / 1000.0
                silence_start = silence_end - silence_ms / 1000.0
                print(
                    f"[silence detector] {silence_ms} ms silence in audio "
                    f"[{silence_start:.2f}s, {silence_end:.2f}s] -- "
                    f"force_eou sent at audio t={silence_end:.2f}s"
                )
                fired = True
        # carry leftover samples (less than one window) into the next call
        self.buffer = samples[n_windows * self.window_samples:]
        return fired

## Run 2 &mdash; with `force_eou`

The request generator inspects each outgoing chunk with `SilenceDetector` and sets `runtime_config={"force_eou": "true"}` on the chunk where the silence boundary is crossed. No SDK changes are required &mdash; `runtime_config` is a generic `map<string, string>` on `StreamingRecognizeRequest`.

In [ ]:
from collections import deque


def force_eou_request_generator(streaming_config, chunks, detector, send_log):
    yield rasr.StreamingRecognizeRequest(streaming_config=streaming_config)
    for chunk in chunks:
        force = detector.process_chunk(chunk)
        if force:
            send_log.append(time.monotonic())
        runtime_config = {"force_eou": "true"} if force else {}
        yield rasr.StreamingRecognizeRequest(
            audio_content=chunk,
            runtime_config=runtime_config,
        )


detector = SilenceDetector(sample_rate=params.framerate)
force_eou_sends = deque()  # wall-clock times of force_eou sends, consumed by collect_finals

print("--- force_eou run ---")
with riva.client.AudioChunkFileIterator(
    str(AUDIO_PATH), CHUNK_FRAMES, delay_callback=riva.client.sleep_audio_length
) as chunks:
    t0 = time.monotonic()
    responses = asr_stub.StreamingRecognize(
        force_eou_request_generator(streaming_config, chunks, detector, force_eou_sends),
        metadata=auth.get_auth_metadata(),
    )
    force_eou_finals = list(collect_finals(responses, t0, force_eou_sends=force_eou_sends))

### Reading the silence-detector log

The detector printed three fires on the force_eou run, but only **two** finals were emitted by the server. Here is what each line means:

- `200 ms silence in audio [0.02s, 0.22s] -- force_eou sent at audio t=0.22s` &mdash; a brief noisy sample at the very start of the clip crossed the RMS threshold for one 20 ms window, marking the detector as "in speech," and the next 200 ms of quiet then satisfied the silence rule. **No final transcript was emitted in response.** This is by design: the cache-aware RNNT decoder honors `force_eou` only after speech has actually been collected for the current utterance. A premature signal is silently dropped, so clients can run aggressive detectors without producing phantom finals.
- `200 ms silence in audio [1.72s, 1.92s]` &mdash; the natural pause after "asleep". The server has buffered real speech, so it finalizes that utterance and emits the first final at wall clock t=2.06s.
- `200 ms silence in audio [4.26s, 4.46s]` &mdash; the trailing pause after "door". The server finalizes the second utterance at wall clock t=4.61s.

Audio timestamps in the silence log are positions **within the wav file** (not wall-clock arrival time at the server). The `t=...s` in the partial / final lines is wall-clock relative to the start of the streaming call. The two should track each other closely because we throttle audio to real time with `sleep_audio_length`.

## Comparison

With server-side endpointing alone, the server waits for its silence threshold (around several hundred milliseconds) before producing a mid-stream final &mdash; for short pauses, that often means the entire input lands as one final at the end of the stream. With `force_eou`, the client triggers finalization after **200 ms** of detected silence, producing an earlier per-utterance final while the stream stays open for the next utterance.

In [ ]:
print(f"{'Run':<22} {'First final at':<18} {'Transcript':<60}")
print("-" * 100)


def first_final(records):
    return records[0] if records else (float('nan'), '<none>')


t_b, txt_b = first_final(baseline_finals)
t_f, txt_f = first_final(force_eou_finals)
print(f"{'baseline':<22} {f'{t_b:.2f}s':<18} {txt_b!r}")
print(f"{'force_eou (200 ms)':<22} {f'{t_f:.2f}s':<18} {txt_f!r}")
print()
if force_eou_finals and baseline_finals:
    print(f"force_eou preempted the server-side EOU by {t_b - t_f:.2f} s on the first utterance.")

## Production considerations

- **RMS is a stand-in.** A 200 ms threshold is aggressive &mdash; natural mid-sentence pauses (breath, hesitation, "uh") often exceed 200 ms. Use a longer threshold, an external VAD, or higher-level signals (push-to-talk, agent decisions) in real applications.
- **Edge-triggered, not level-triggered.** `force_eou` is best modeled as a one-shot finalize event. The detector here fires once per speech&rarr;silence transition and suppresses repeats; sending `force_eou=true` on every chunk during a silence is harmless on the server but conceptually wrong.
- **Model gating.** `force_eou` is currently honored only by cache-aware RNNT models (e.g., Nemotron ASR Streaming). Other decoders silently ignore the flag.
- **Detection is decoupled from chunk size.** `SilenceDetector` buffers samples internally, so client chunk size doesn't constrain silence resolution &mdash; smaller chunks just mean more requests on the wire.
- **`force_eou` latency depends on chunk sizes.** The `+ms after force_eou` delay reported above is bounded primarily by the **server's processing chunk size**: the server can only act on the flag at its next chunk boundary. When the client chunk size is smaller than the server's, the wait time falls in the range `[0, server_chunk_size]` (plus network RTT and a small decoder-flush overhead). Larger client chunks introduce their own buffering delay before the flag even reaches the server.

## Go deeper into Riva capabilities

### Additional Riva tutorials

Check out more Riva tutorials [here](https://github.com/nvidia-riva/tutorials) for advanced features &mdash; word boosting, speaker diarization, custom vocabularies, and pipeline customization.

### Additional resources

- [Riva ASR NIM documentation](https://docs.nvidia.com/nim/speech/latest/asr/index.html)
- [Riva ASR NIM support matrix](https://docs.nvidia.com/nim/speech/latest/reference/support-matrix/asr.html)
- [Streaming ASR proto reference](https://docs.nvidia.com/nim/speech/latest/reference/api-references/asr/protos.html#riva-proto-riva-asr-proto)